# Music Annotation Project — Notebook 71
## Deduplicated Whole-Page Masks and Fresh Engine Rerun

**Runtime:** CPU. Run all in Google Colab.

Notebook 71 selects one authoritative whole-page mask per observed page, removes byte-identical duplicates by semantic binary SHA-256, and freshly reruns:

1. **Notebook 09 / frozen V7.8 geometry_t80** from the observed page and deduplicated mask.
2. **V8.8 registered replay** by reconstructing the registered residual mask from the identical M3 canvas plus the frozen SAFE_REMOVE and historical code-2 exclusion decisions, then rerunning Telea radius 3.

No clean truth is loaded. No prior corrected page is copied as an engine result. V8.8 stops for a page unless the deduplicated prepared mask is identical to the frozen Notebook 53R3 M3 mask.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow


## 1. Configuration and helpers

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform
import cv2, numpy as np, pandas as pd
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
OUT=ROOT/'notebook71_deduplicated_fresh_rerun'/'runs'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook71')
T=OUT/'tables'; P=OUT/'pages'; A=OUT/'audit'; U=OUT/'upload'
for p in [OUT,T,P,A,U]: p.mkdir(parents=True,exist_ok=False)
OBSERVED_DIR_OVERRIDE=None
OBSERVED_DIR_CANDIDATES=[ROOT/'production_pilot'/'incoming',ROOT/'aligned',ROOT/'source'/'annotated_only']
MASK_SOURCES=[('pilot_reviewed',ROOT/'production_pilot'/'reviewed_masks'),('hitl_corrected',ROOT/'human_in_the_middle'/'corrected_masks'),('project_corrected',ROOT/'masks'/'corrected'),('pilot_run_masks',ROOT/'production_pilot'/'runs'),('hitl_drafts',ROOT/'human_in_the_middle'/'draft_masks')]
N53R3_RUN_OVERRIDE=None
N53R3_ROOTS=[ROOT/'observed_print_classifier_v8_8b_regression_compatibility',ROOT/'v8_8b_notebook51r2_regression_compatibility',ROOT/'observed_print_classifier_v8_8b']
DILATION=2; MARGIN=48; GTH=80; BAND=4; RADIUS=3
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def req(x,m):
 if not x: raise RuntimeError(m)
def sha_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def ah(a):
 a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def mh(a): return ah((a>0).astype(np.uint8))
def read(p):
 a=cv2.imread(str(p),0); req(a is not None,f'Unreadable: {p}'); return a
def files(d): return sorted(p for p in Path(d).rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED) if Path(d).exists() else []
def atomic(p,v):
 q=Path(str(p)+'.tmp'); q.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); q.replace(p)
def binary(a): return np.where(a>0,255,0).astype(np.uint8)
def prep(a): return cv2.dilate(binary(a),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*DILATION+1,2*DILATION+1)))
def immutable(o,c,m):
 x=o.copy(); x[m>0]=c[m>0]; req(np.array_equal(x[m==0],o[m==0]),'Outside mask drift'); return x
def boxes(mask):
 n,l,s,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8); H,W=mask.shape; b=[]
 for i in range(1,n):
  x,y,w,h,a=s[i]
  if a>=8:b.append((max(0,x-MARGIN),max(0,y-MARGIN),min(W,x+w+MARGIN),min(H,y+h+MARGIN)))
 changed=True
 while changed:
  changed=False; out=[]
  while b:
   x0,y0,x1,y1=b.pop(); rest=[]
   for a0,b0,a1,b1 in b:
    if not (x1<a0 or a1<x0 or y1<b0 or b1<y0): x0=min(x0,a0);y0=min(y0,b0);x1=max(x1,a1);y1=max(y1,b1);changed=True
    else: rest.append((a0,b0,a1,b1))
   b=rest;out.append((x0,y0,x1,y1))
  b=out
 return b
def telea(obs,mask):
 out=obs.copy()
 for x0,y0,x1,y1 in boxes(mask):
  im=obs[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]; cand=cv2.inpaint(im,mk,RADIUS,cv2.INPAINT_TELEA); q=out[y0:y1,x0:x1]; q[mk>0]=cand[mk>0]
 return immutable(obs,out,mask)
def surface(obs,mask):
 v=(mask==0)&(obs>=200); n=cv2.GaussianBlur(obs.astype(np.float32)*v,(0,0),45); d=cv2.GaussianBlur(v.astype(np.float32),(0,0),45); f=cv2.GaussianBlur(obs,(0,0),45); return np.clip(np.where(d>.02,n/np.maximum(d,1e-4),f),0,255).astype(np.uint8)
def supports(obs,mask):
 dark=(obs<190)&(mask==0); h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0; v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
 n,l,s,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); c=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=s[i]
  if 3<=a<=650 and w<=40 and hg<=40:c[l==i]=True
 k=np.ones((2*BAND+1,2*BAND+1),np.uint8); return *[cv2.dilate(z.astype(np.uint8),k)>0 for z in (h,v,c)],
def geometry(obs,mask,t):
 paper=surface(obs,mask); norm=cv2.divide(obs,np.maximum(paper,1),scale=255); h,v,c=supports(obs,mask); m=mask>0; preserve=m&(h|v|c)&(obs<200); remove=m&~preserve&(obs>=GTH)&(norm>=GTH); amb=m&~preserve&~remove
 out=obs.copy(); out[remove]=paper[remove];out[amb]=t[amb];out[preserve]=obs[preserve]
 return immutable(obs,out,mask),{'preserve':preserve,'remove':remove,'ambiguous':amb,'horizontal':h,'vertical':v,'compact':c}
print('Output:',OUT)


## 2. Deduplicate masks and resolve frozen V8.8 source

In [ ]:
obs_dir=Path(OBSERVED_DIR_OVERRIDE) if OBSERVED_DIR_OVERRIDE else next((d for d in OBSERVED_DIR_CANDIDATES if files(d)),None); req(obs_dir,'No observed pages')
obs={}
for p in files(obs_dir): obs.setdefault(p.stem,[]).append(p)
for k,v in obs.items(): req(len(v)==1,f'{k}: duplicate observed pages')
obs={k:v[0] for k,v in obs.items()}
rows=[]
for pri,(name,d) in enumerate(MASK_SOURCES):
 for p in files(d):
  page=p.stem
  for s in ['_authoritative','_corrected','_mask','_draft']:
   if page.endswith(s): page=page[:-len(s)]
  if page not in obs: continue
  m=read(p); o=read(obs[page]); rows.append({'page':page,'source':name,'priority':pri,'path':str(p),'valid':m.shape==o.shape,'hash':mh(m) if m.shape==o.shape else None,'pixels':int((m>0).sum()) if m.shape==o.shape else None})
idx=pd.DataFrame(rows); req(len(idx),'No matched masks'); idx.to_csv(T/'all_mask_candidates.csv',index=False)
chosen=[];dups=[];conf=[]
for page,g in idx[idx.valid].groupby('page'):
 g=g.sort_values(['priority','path']); first=g.iloc[0]; same=g[g.hash==first.hash]; other=g[g.hash!=first.hash]
 for _,r in same.iloc[1:].iterrows(): dups.append({'page':page,'kept':first.path,'duplicate':r.path,'hash':r.hash})
 for _,r in other.iterrows(): conf.append({'page':page,'selected':first.path,'conflict':r.path,'selected_hash':first.hash,'conflict_hash':r.hash})
 chosen.append({'page':page,'source':first.source,'path':first.path,'hash':first.hash,'duplicate_count':len(same)-1,'conflict_count':len(other)})
chosen=pd.DataFrame(chosen); req(len(chosen),'No chosen masks'); chosen.to_csv(T/'selected_authoritative_masks.csv',index=False); pd.DataFrame(dups).to_csv(T/'byte_identical_duplicates.csv',index=False); pd.DataFrame(conf).to_csv(T/'nonidentical_conflicts.csv',index=False)
def complete(p): return p.is_dir() and (p/'experiment_identity.json').exists() and (p/'pages').exists()
if N53R3_RUN_OVERRIDE:N53=Path(N53R3_RUN_OVERRIDE)
else:
 hits=[]
 for r in N53R3_ROOTS:
  if r.exists():hits += [p for p in r.rglob('*') if complete(p)]
 N53=max(hits,key=lambda p:p.stat().st_mtime) if hits else None
req(N53,'No N53R3 run; set override'); identity=json.loads((N53/'experiment_identity.json').read_text()); req(identity.get('version')=='53R3-v8.8b-notebook51R2-regression-compatibility','Wrong N53 version'); req(identity.get('mode')=='REGISTERED_REPLAY','N53 not registered replay')
print('Masks:',len(chosen),'duplicates:',len(dups),'conflicts:',len(conf),'N53:',N53)


## 3. Fresh engine rerun

In [ ]:
done=[];failed=[]
for _,r in chosen.sort_values('page').iterrows():
 page=str(r.page); d=P/page; d.mkdir(parents=True,exist_ok=False)
 try:
  o=read(obs[page]); seed=read(r.path); m3=prep(seed); t=telea(o,m3); n09,routes=geometry(o,m3,t)
  src=N53/'pages'/page; req(src.exists(),f'{page}: absent from N53 roster'); frozen_m3=read(src/'m3.png')>0; req(ah(frozen_m3.astype(np.uint8))==ah((m3>0).astype(np.uint8)),f'{page}: deduplicated M3 differs from frozen M3')
  sr=read(src/'safe_remove.png')>0; req(np.all(~sr|frozen_m3),f'{page}: bad SAFE_REMOVE'); pp=src/'safe_protect.png'; ex=read(pp)>0 if pp.exists() else np.zeros_like(sr); req(np.all(~ex|frozen_m3),f'{page}: bad code2 exclusion')
  m5=frozen_m3&~sr&~ex; v88=telea(o,m5.astype(np.uint8)*255); req(np.array_equal(n09[m3==0],o[m3==0]) and np.array_equal(v88[m3==0],o[m3==0]),f'{page}: outside drift')
  diff=n09!=v88
  for name,a in [('observed.png',o),('authoritative_mask_undilated.png',binary(seed)),('m3_dilation2.png',m3),('notebook09_fresh.png',n09),('v8_8_registered_replay_fresh.png',v88),('safe_remove.png',sr.astype(np.uint8)*255),('registered_code2_exclusion.png',ex.astype(np.uint8)*255),('m5_registered_replay.png',m5.astype(np.uint8)*255),('engine_difference.png',diff.astype(np.uint8)*255)]: cv2.imwrite(str(d/name),a)
  np.savez_compressed(d/'fresh_arrays.npz',m3=m3,safe_remove=sr.astype(np.uint8),registered_exclusion=ex.astype(np.uint8),m5=m5.astype(np.uint8),notebook09=n09,v88=v88,**{f'n09_{k}':v.astype(np.uint8) for k,v in routes.items()})
  rec={'page':page,'mask_source':r.source,'mask_path':r.path,'m3_hash':mh(m3),'safe_remove_hash':mh(sr),'m5_hash':mh(m5),'notebook09_hash':ah(n09),'v88_hash':ah(v88),'difference_pixels':int(diff.sum()),'fresh_notebook09':True,'fresh_v88_reconstruction':True,'frozen_v88_decisions_reused':True,'telea_radius':RADIUS}; atomic(d/'page_manifest.json',rec);done.append(rec)
 except Exception as e: failed.append({'page':page,'error':str(e)});atomic(d/'failure.json',{'page':page,'error':str(e)})
res=pd.DataFrame(done);fail=pd.DataFrame(failed);res.to_csv(T/'fresh_engine_results.csv',index=False);fail.to_csv(T/'failures.csv',index=False)
print('Completed',len(res),'Failed',len(fail));display(fail) if len(fail) else None


## 4. Replay, verdict and upload bundle

In [ ]:
re=[]
for _,r in res.iterrows():
 d=P/r.page;o=read(d/'observed.png');m3=read(d/'m3_dilation2.png');sr=read(d/'safe_remove.png')>0;ex=read(d/'registered_code2_exclusion.png')>0;n09,_=geometry(o,m3,telea(o,m3));v88=telea(o,((m3>0)&~sr&~ex).astype(np.uint8)*255);re.append({'page':r.page,'notebook09_exact':ah(n09)==r.notebook09_hash,'v88_exact':ah(v88)==r.v88_hash})
re=pd.DataFrame(re);re.to_csv(A/'deterministic_replay.csv',index=False)
checks={'selected_masks':len(chosen)>0,'all_pages_completed':len(res)==len(chosen),'no_failures':len(fail)==0,'exact_replay':bool(len(re) and re[['notebook09_exact','v88_exact']].all().all())}; first=next((k for k,v in checks.items() if not v),None);verdict='PASS' if first is None else 'FAIL'
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'71-deduplicated-whole-page-fresh-engine-rerun','verdict':verdict,'first_failed_check':first,'checks':checks,'observed_dir':str(obs_dir),'n53r3_run':str(N53),'pages':len(chosen),'completed':len(res),'duplicates_removed':len(dups),'conflicts_recorded':len(conf)};atomic(OUT/'notebook71_verdict.json',summary);(OUT/'executive_summary.txt').write_text('\n'.join(['NOTEBOOK 71',f'VERDICT: {verdict}',f'Pages: {len(chosen)}',f'Completed: {len(res)}',f'Duplicates removed: {len(dups)}',f'Conflicts recorded: {len(conf)}',* [f'{k}: {v}' for k,v in checks.items()]]))
items=[OUT/'executive_summary.txt',OUT/'notebook71_verdict.json',T/'selected_authoritative_masks.csv',T/'byte_identical_duplicates.csv',T/'nonidentical_conflicts.csv',T/'fresh_engine_results.csv',T/'failures.csv',A/'deterministic_replay.csv']
for page in res.page.tolist():
 d=P/page;items += [d/x for x in ['observed.png','authoritative_mask_undilated.png','m3_dilation2.png','notebook09_fresh.png','v8_8_registered_replay_fresh.png','engine_difference.png','page_manifest.json']]
manifest=[]
for s in items:
 req(s.exists(),f'Missing upload item {s}'); q=U/s.relative_to(OUT);q.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(s,q);manifest.append({'relative_path':str(q.relative_to(U)),'bytes':q.stat().st_size,'sha256':sha_file(q)})
pd.DataFrame(manifest).to_csv(U/'upload_manifest.csv',index=False);(U/'README.txt').write_text('Start with executive_summary.txt, notebook71_verdict.json, fresh_engine_results.csv, and per-page engine_difference.png.\n');archive=shutil.make_archive(str(OUT/'notebook71_upload_bundle'),'zip',U)
print(json.dumps(summary,indent=2));print('Bundle:',archive);print('COMPLETE:',OUT)
